In [ ]:
from google.cloud import bigquery

DATASET_NAME = "" # @param {"type":"string","placeholder":"Enter your dataset name"}

bq_client = bigquery.Client()

In [ ]:
# @title Get storage billing model recommendations

df_datasets = bq_client.query_and_wait(f"""
SELECT
  project_id,
  target_resource,
  CONCAT(
    'ALTER SCHEMA `',
    SAFE.STRING(operations[0].resource_name),
    '` SET OPTIONS(storage_billing_model = "',
    SAFE.STRING(operations[0].recommended_storage_billing_model),
    '");'
  ) AS ddl,
  SAFE.FLOAT64(overview.cost_30d) AS cost_usd_30d,
  SAFE.FLOAT64(overview.savings_30d) AS savings_usd_30d
FROM `masthead-prod`.{DATASET_NAME}.insights
WHERE subtype = 'Storage billing model'
ORDER BY
  FLOAT64(overview.savings_30d) DESC
""").to_dataframe()

df_datasets.head(10)

In [ ]:
# @title Apply recommendations using DDL statements

datasets_to_update = df_datasets.loc[df_datasets['savings_usd_30d'] > 100]

for _, row in datasets_to_update.iterrows():
    print(row["ddl"])
    # bq_client.query(row["ddl"]).result()